# 02 — Model training walkthrough
Loads gold-tier features, trains XGBoost and LightGBM, evaluates on a hold-out set, and produces SHAP summary plots.

In [ ]:
import sys, pandas as pd
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
from src.utils.config import load_config
from src.models import XGBoostTrainer, LightGBMTrainer
from src.evaluation import classification_metrics
from sklearn.model_selection import train_test_split
cfg = load_config('../configs/pipeline.yaml')
features = pd.read_parquet('../data/gold/features.parquet')
features.head()

In [ ]:
from src.features.encoders import CATEGORICAL_COLS, fit_one_hot, apply_one_hot
from sklearn.model_selection import train_test_split
y = features['final_result_binary'].astype(int)
X = features.drop(columns=['final_result_binary','id_student'])
cat = [c for c in CATEGORICAL_COLS if c in X.columns]
enc = fit_one_hot(X, cat)
X = apply_one_hot(X, enc, cat).fillna(0)
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,stratify=y,random_state=42)


In [ ]:
trainer = XGBoostTrainer(cfg.models['xgboost'])
trainer.fit(X_train,y_train,X_val=X_test,y_val=y_test)
scores = trainer.predict_proba(X_test)
classification_metrics(y_test.to_numpy(),scores)